# ChromaDB Health Report

Inspect persistent ChromaDB collections and basic metadata health.

⚠️ **IMPORTANT**: If you recently ran code ingestion with `--reset`, restart the kernel before running to ensure fresh data.

In [ ]:
# Connect to ChromaDB PersistentClient
import os, pathlib
from datetime import datetime

from chromadb import PersistentClient
from scripts.ingest.ingest_config import IngestConfig

from scripts.utils.db_factory import get_default_vector_path, get_vector_client
from pathlib import Path

config = IngestConfig()

# Get the correct ChromaDB path using factory pattern
PersistentClient_class, using_sqlite = get_vector_client(prefer="chroma")
chroma_path = get_default_vector_path(Path(config.rag_data_path), using_sqlite)

print(f"rag_data_path: {config.rag_data_path}")
print(f"ChromaDB path: {chroma_path}")
print()

# Connect to ChromaDB
client = PersistentClient_class(path=str(chroma_path))

collections = client.list_collections()
print(f'⏰ Timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'Using SQLite fallback: {using_sqlite}')
print(f'Collections found: {len(collections)}')
for c in collections:
    print(f'  - {c.name}: {c.count():,} chunks')

In [ ]:
# Get chunk collection
chunk_name = os.getenv('CHUNK_COLLECTION_NAME', 'governance_docs_chunks')
chunk_col = client.get_or_create_collection(chunk_name)
chunk_count = chunk_col.count()

print(f'Chunk collection: {chunk_name}')
print(f'Total chunks: {chunk_count:,}')
if chunk_count == 0:
    print('⚠️  Empty collection - if recently ingested with --reset, restart kernel')

In [ ]:
# Sample metadata health: chunks per doc/file distribution (sample 500)
if chunk_count == 0:
    print('⚠️  Cannot analyse empty collection')
else:
    from collections import Counter
    
    sample = chunk_col.get(include=['metadatas'], limit=500)
    doc_ids = [m.get('doc_id') for m in sample.get('metadatas', []) if m.get('doc_id')]
    counts = Counter(doc_ids)
    
    # Check for language metadata (code ingestion)
    languages = [m.get('language') for m in sample.get('metadatas', []) if m.get('language')]
    lang_counts = Counter(languages)
    
    if counts:
        vals = list(counts.values())
        print(f'Sampled docs: {len(counts)}')
        print(f'Chunks per doc: avg {sum(vals)/len(vals):.1f}, min {min(vals)}, max {max(vals)}')
    
    if lang_counts:
        print(f'\nLanguages detected:')
        for lang, count in lang_counts.most_common(10):
            print(f'  {lang}: {count}')

In [ ]:
# Check metadata completeness
if chunk_count == 0:
    print('⚠️  Cannot check metadata - collection empty')
else:
    required_keys = ('doc_id', 'language', 'service_name')
    missing_counts = {k: 0 for k in required_keys}
    total_sample = len(sample.get('metadatas', []))
    
    if total_sample == 0:
        print('No metadata in sample')
    else:
        for m in sample.get('metadatas', []):
            for k in required_keys:
                if m.get(k) is None or m.get(k) == '':
                    missing_counts[k] += 1
        
        print(f'Metadata completeness (sample of {total_sample} chunks):')
        for k in required_keys:
            missing = missing_counts[k]
            present = total_sample - missing
            pct = (present / total_sample * 100) if total_sample > 0 else 0
            status = '✓' if missing == 0 else '⚠️'
            print(f'  {k:20s}: {present:4d}/{total_sample} ({pct:5.1f}%) {status}')

In [ ]:
# Health analysis: content type and service detection
if chunk_count == 0 or 'sample' not in locals():
    print('⚠️  Cannot analyse - collection not available')
else:
    import json
    from collections import Counter
    
    print('🔄 Analysing collection...')
    
    code_chunks = 0
    doc_chunks = 0
    service_names = Counter()
    endpoints_found = 0
    
    metadatas = sample.get('metadatas', [])
    for m in metadatas:
        if m:
            if m.get('language'):
                code_chunks += 1
            if m.get('doc_type'):
                doc_chunks += 1
            if m.get('service_name'):
                service_names[m.get('service_name')] += 1
            if m.get('endpoints'):
                endpoints_found += 1
    
    print(f'📊 Content Analysis:')
    print(f'  Code chunks: {code_chunks:,}')
    print(f'  Document chunks: {doc_chunks:,}')
    print(f'  Chunks with endpoints: {endpoints_found:,}')
    
    if service_names:
        print(f'\n📦 Top services detected:')
        for svc_name, count in service_names.most_common(10):
            print(f'  {svc_name}: {count}')
    
    print(f'\n✓ Analysis complete')

In [ ]:
# Chunk ordering and structure check
if chunk_count == 0 or 'sample' not in locals():
    print('⚠️  Cannot analyse - collection not available')
else:
    from collections import defaultdict
    
    print('Checking chunk structure...')
    
    ids = sample.get('ids', [])
    metadatas = sample.get('metadatas', [])
    
    # Find chunks from same doc
    chunks_by_doc = defaultdict(list)
    for i, m in enumerate(metadatas):
        doc_id = m.get('doc_id')
        if doc_id:
            chunks_by_doc[doc_id].append((i, m, ids[i] if i < len(ids) else None))
    
    print(f'Found {len(chunks_by_doc)} unique docs in sample')
    
    # Show sample multi-chunk doc
    for doc_id, chunks in list(chunks_by_doc.items())[:3]:
        if len(chunks) > 1:
            print(f'\nDoc \"{doc_id[:50]}...\" has {len(chunks)} chunks')
            for idx, (pos, meta, chroma_id) in enumerate(chunks[:3]):
                print(f'  Chunk #{idx}: {chroma_id}')
            break

In [ ]:
# Cross-repository service clustering
if chunk_count == 0 or 'sample' not in locals():
    print('⚠️  Cannot analyse - collection not available')
else:
    from collections import defaultdict
    
    print('🔍 Cross-Repo Service Analysis...')
    
    # Query larger sample to detect cross-repo patterns
    large_sample = chunk_col.get(include=['metadatas'], limit=1000)
    metadatas = large_sample.get('metadatas', [])
    
    # Group services by repository
    service_repos = defaultdict(set)
    repo_services = defaultdict(set)
    repo_files = defaultdict(set)
    
    for m in metadatas:
        if m:
            service = m.get('service_name')
            repo = m.get('repository')
            project = m.get('project')
            file_id = m.get('doc_id')
            
            if service and repo:
                service_repos[service].add(repo)
                repo_services[repo].add(service)
            if repo and file_id:
                repo_files[repo].add(file_id.split('/')[0] if '/' in file_id else file_id)
    
    print(f'📊 Repository Coverage:')
    print(f'  Repositories found: {len(repo_services)}')
    for repo, services in sorted(repo_services.items())[:5]:
        print(f'    {repo}: {len(services)} unique services')
    
    print(f'\n🔗 Cross-Repo Services (appearing in multiple repos):')
    cross_repo_services = {svc: repos for svc, repos in service_repos.items() if len(repos) > 1}
    if cross_repo_services:
        for service, repos in sorted(cross_repo_services.items(), key=lambda x: -len(x[1]))[:10]:
            print(f'  {service}: {", ".join(sorted(repos))}')
    else:
        print('  None (each service appears in only one repo)')
    
    print(f'\n✓ Cross-repo analysis complete')

## Notes

- Use environment variables to target different collections
- Increase `limit` in cell 4 for deeper sampling on large collections
- For code ingestion: look for `language`, `service_name`, `endpoints` fields
- For document ingestion: look for `doc_type`, `doc_id`, `version` fields

## Cross-Repository Analysis

Cell 8 shows **cross-repo service clustering**:
- Identifies services appearing in multiple repositories
- Flags potential code duplication or shared libraries
- Services in multiple repos create clustering edges in the consistency graph with severity 0.5
- Use this to detect:
  - Duplicate implementations (same service name in different repos)
  - Shared libraries (intentional cross-repo usage)
  - Services needing consolidation